# US Jobs — Data Cleaning Pipeline

**Input :** `us_jobs_parsed.csv`  — raw output of Gemini 2.5 Flash Pro Proparsing
**Output:** `us_jobs_final.csv`  — all sources, quality-filtered & normalized

### Sources retained
Both **LinkedIn** and **USAJobs** rows are kept when they pass quality checks.
Low-quality USAJobs rows (missing hard_skills, wrong category) are dropped naturally
by the quality filters below — no hard source exclusion.

### Cleaning steps
| # | Step |
|---|------|
| 1 | Fix `job_id` — strip float `.0` suffix |
| 2 | Fix `posted_date` → `YYYY-MM-DD` |
| 3 | Drop rows with missing `hard_skills` |
| 4 | Drop rows with missing `parsed_title` |
| 5 | Salary — nullify zeros, compute `salary_midpoint` |
| 6 | Strip whitespace from all text columns |
| 7 | Drop `job_category == 'other'` |

> **Note on deduplication:** `us_jobs_parsed.csv` is generated from `us_jobs_deduped.csv`
> which has already been deduplicated by the embedding pipeline. No dedup step needed here.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

RAW_PARSED = 'data/us_jobs_parsed.csv'
OUTPUT_CSV = 'data/us_jobs_final.csv'

df_raw = pd.read_csv(RAW_PARSED, encoding='utf-8-sig')
print(f'Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print()
print('Source breakdown (raw):')
print(df_raw['source'].value_counts().to_string())
print()
print('Job category breakdown (raw):')
print(df_raw['job_category'].value_counts().to_string())
df_raw.head(3)

Loaded: 2,445 rows × 20 columns

Source breakdown (raw):
source
LinkedIn    1981
USAJobs      464

Job category breakdown (raw):
job_category
software-eng      853
data-science      424
other             416
devops            265
infrastructure    141
security          135
product           121
design             90


,source,job_id,job_title,company,location,posted_date,url,salary_min_raw,salary_max_raw,parsed_title,seniority,min_years_exp,employment_type,remote_status,salary_min_annual,salary_max_annual,hard_skills,soft_skills,education,job_category
0,LinkedIn,4.396161e+09,Frontend Software Engineer,Cabana,"Palo Alto, CA",31-03-26,https://www.linkedin.com/jobs/view/4396160922,NaN,NaN,Frontend Software Engineer,mid,2.0,full-time,onsite,135000.0,200000.0,"JavaScript, TypeScript, frontend frameworks, web applications, frontend arch...","ownership, critical thinking, product sense, problem-solving, collaboration,...",unknown,software-eng
1,LinkedIn,4.395453e+09,Software Engineer,Epsilon,"Westminster, CO",07-04-26,https://www.linkedin.com/jobs/view/4395453046,NaN,NaN,Software Engineer,junior,1.0,full-time,unknown,73500.0,136500.0,"Java, Python, SQL, relational databases, Unix, Unix Shell Scripts, Hadoop, H...","ownership, teamwork, collaboration, autonomy, growth mindset, continuous imp...",bachelor,software-eng
2,LinkedIn,4.364164e+09,Software Engineer 5 - Ads Member Experience,Netflix,"Los Gatos, CA",29-03-26,https://www.linkedin.com/jobs/view/4364164472,NaN,NaN,Software Engineer 5 - Ads Member Experience,senior,NaN,full-time,unknown,NaN,NaN,"ad formats, ad delivery systems, experimentation, ad creatives, native ads, ...","collaboration, creativity, innovation",unknown,software-eng


In [3]:
def empty_counts(df):
    is_empty = df.isnull() | df.apply(lambda col: col.astype(str).str.strip() == '')
    empty = is_empty.sum()
    pct   = (empty / len(df) * 100).round(1)
    return pd.DataFrame({'empty': empty, 'pct_%': pct}).sort_values('pct_%', ascending=False)

print('=== Empty / Null counts — RAW ===')
print(empty_counts(df_raw).to_string())

=== Empty / Null counts — RAW ===
                   empty  pct_%
salary_max_raw      1981   81.0
salary_min_raw      1981   81.0
salary_max_annual   1551   63.4
salary_min_annual   1541   63.0
min_years_exp       1289   52.7
job_id               464   19.0
soft_skills          361   14.8
hard_skills          170    7.0
parsed_title          34    1.4
company                0    0.0
source                 0    0.0
job_title              0    0.0
seniority              0    0.0
location               0    0.0
posted_date            0    0.0
url                    0    0.0
employment_type        0    0.0
remote_status          0    0.0
education              0    0.0
job_category           0    0.0


In [4]:
df = df_raw.copy()
drop_log = {}

### Step 1 — Fix job_id

In [5]:
def clean_job_id(val):
    if pd.isna(val) or str(val).strip() in ('', 'nan'):
        return ''
    try:
        return str(int(float(str(val).strip())))
    except (ValueError, OverflowError):
        return str(val).strip()

df['job_id'] = df['job_id'].apply(clean_job_id)
print('Sample :', df['job_id'].head(5).tolist())
print('Still ends .0:', df['job_id'].str.endswith('.0').sum())
print('Empty job_id (expected for USAJobs):', (df['job_id'] == '').sum())

Sample : ['4396160922', '4395453046', '4364164472', '4397376471', '4328991042']
Still ends .0: 0
Empty job_id (expected for USAJobs): 464


### Step 2 — Fix posted_date → YYYY-MM-DD

In [6]:
df['posted_date'] = pd.to_datetime(df['posted_date'], format='%d-%m-%y', errors='coerce')
print(f'Invalid dates : {df["posted_date"].isnull().sum()}')
print(f'Range         : {df["posted_date"].min().date()} → {df["posted_date"].max().date()}')

Invalid dates : 0
Range         : 2025-06-18 → 2026-04-15


### Step 3 — Drop rows with missing hard_skills

In [7]:
before = len(df)
mask = df['hard_skills'].isna() | (df['hard_skills'].astype(str).str.strip() == '')
dropped = df[mask][['source', 'job_category']].value_counts()
df = df[~mask].copy()
drop_log['3_no_hard_skills'] = before - len(df)
print(f'Dropped: {drop_log["3_no_hard_skills"]}  |  Remaining: {len(df):,}')
print('Dropped by source/category:')
print(dropped.to_string())

Dropped: 170  |  Remaining: 2,275
Dropped by source/category:
source    job_category
USAJobs   other           148
          data-science      8
LinkedIn  software-eng      7
USAJobs   software-eng      5
          design            1
          security          1


### Step 4 — Drop rows with missing parsed_title

In [8]:
before = len(df)
mask = df['parsed_title'].isna() | (df['parsed_title'].astype(str).str.strip() == '')
df = df[~mask].copy()
drop_log['4_no_parsed_title'] = before - len(df)
print(f'Dropped: {drop_log["4_no_parsed_title"]}  |  Remaining: {len(df):,}')

Dropped: 0  |  Remaining: 2,275


### Step 5 — Salary cleaning

In [9]:
# 5a: Nullify zero salaries
for col in ['salary_min_annual', 'salary_max_annual']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    n_zero = (df[col] == 0.0).sum()
    df.loc[df[col] == 0.0, col] = np.nan
    print(f'{col}: {n_zero} zeros → NaN')

# 5b: salary_midpoint
df['salary_midpoint'] = (df['salary_min_annual'] + df['salary_max_annual']) / 2
print(f'\nsalary_midpoint non-null: {df["salary_midpoint"].notna().sum()} rows ({df["salary_midpoint"].notna().mean()*100:.1f}%)')
print('Note: salary values are LLM-inferred from job description text.')

salary_min_annual: 2 zeros → NaN
salary_max_annual: 2 zeros → NaN

salary_midpoint non-null: 745 rows (32.7%)
Note: salary values are LLM-inferred from job description text.


### Step 6 — Strip whitespace from all text columns

In [10]:
str_cols = df.select_dtypes(include='object').columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip()
print(f'Stripped {len(str_cols)} string columns')

Stripped 14 string columns


### Step 7 — Drop job_category == 'other'

In [11]:
before = len(df)
dropped_other = df[df['job_category'].str.lower() == 'other'][['source']].value_counts()
df = df[df['job_category'].str.lower() != 'other'].copy()
drop_log['7_drop_other_cat'] = before - len(df)
print(f'Dropped: {drop_log["7_drop_other_cat"]}  |  Remaining: {len(df):,}')
print('Dropped "other" by source:')
print(dropped_other.to_string())

Dropped: 268  |  Remaining: 2,007
Dropped "other" by source:
source  
USAJobs     231
LinkedIn     37


---
### Cleaning Summary

In [12]:
print('=== CLEANING SUMMARY ===')
print(f'  Raw rows       : {len(df_raw):,}')
for step, n in drop_log.items():
    print(f'  {step:<30}: -{n}')
print(f'  Final rows     : {len(df):,}')
print(f'  Rows retained  : {100 * len(df) / len(df_raw):.1f}%')
print()
print('Source breakdown (final):')
print(df['source'].value_counts().to_string())
print()
print('Job category breakdown (final):')
print(df['job_category'].value_counts().to_string())
print()
print('=== Empty value counts — CLEANED ===')
print(empty_counts(df).to_string())

=== CLEANING SUMMARY ===
  Raw rows       : 2,445
  3_no_hard_skills              : -170
  4_no_parsed_title             : -0
  7_drop_other_cat              : -268
  Final rows     : 2,007
  Rows retained  : 82.1%

Source breakdown (final):
source
LinkedIn    1937
USAJobs       70

Job category breakdown (final):
job_category
software-eng      841
data-science      416
devops            265
infrastructure    141
security          134
product           121
design             89

=== Empty value counts — CLEANED ===
                   empty  pct_%
salary_max_raw      1937   96.5
salary_min_raw      1937   96.5
salary_midpoint     1501   74.8
salary_max_annual   1484   73.9
salary_min_annual   1473   73.4
min_years_exp        874   43.5
soft_skills           88    4.4
job_id                70    3.5
source                 0    0.0
location               0    0.0
company                0    0.0
job_title              0    0.0
url                    0    0.0
employment_type        0    0.0

---
### Export to us_jobs_final.csv

In [13]:
col_order = [
    'job_id', 'source', 'url',
    'job_title', 'parsed_title', 'company', 'location', 'posted_date',
    'seniority', 'min_years_exp', 'employment_type', 'remote_status',
    'salary_min_annual', 'salary_max_annual', 'salary_midpoint',
    'hard_skills', 'soft_skills', 'education', 'job_category',
]

df_out = df[col_order].copy()
df_out['posted_date'] = df_out['posted_date'].dt.strftime('%Y-%m-%d')
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'Exported → {OUTPUT_CSV}')
print(f'  Rows    : {len(df_out):,}')
print(f'  Columns : {df_out.shape[1]}')
print(f'  {df_out.columns.tolist()}')

Exported → data/us_jobs_final.csv
  Rows    : 2,007
  Columns : 19
  ['job_id', 'source', 'url', 'job_title', 'parsed_title', 'company', 'location', 'posted_date', 'seniority', 'min_years_exp', 'employment_type', 'remote_status', 'salary_min_annual', 'salary_max_annual', 'salary_midpoint', 'hard_skills', 'soft_skills', 'education', 'job_category']


---
### Verification

In [14]:
dv = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig', dtype={'job_id': str})
checks = {
    'job_id — no .0 suffix'     : dv['job_id'].astype(str).str.endswith('.0').sum() == 0,
    'job_category — no other'   : (dv['job_category'].str.lower() == 'other').sum() == 0,
    'hard_skills — none empty'  : (dv['hard_skills'].isna() | (dv['hard_skills'] == '')).sum() == 0,
    'parsed_title — none empty' : (dv['parsed_title'].isna() | (dv['parsed_title'] == '')).sum() == 0,
    'salary_min — no zeros'     : (dv['salary_min_annual'] == 0).sum() == 0,
    'no salary_is_explicit col' : 'salary_is_explicit' not in dv.columns,
}
for k, v in checks.items():
    print(f'  {"✓" if v else "✗"} {k}')
all_ok = all(checks.values())
print()
print('✓ All checks passed!' if all_ok else '✗ WARNING — some checks failed')
print()
print('Source distribution in final CSV:')
print(dv['source'].value_counts().to_string())
print()
dv.head(3)

  ✓ job_id — no .0 suffix
  ✓ job_category — no other
  ✓ hard_skills — none empty
  ✓ parsed_title — none empty
  ✓ salary_min — no zeros
  ✓ no salary_is_explicit col

✓ All checks passed!

Source distribution in final CSV:
source
LinkedIn    1937
USAJobs       70



,job_id,source,url,job_title,parsed_title,company,location,posted_date,seniority,min_years_exp,employment_type,remote_status,salary_min_annual,salary_max_annual,salary_midpoint,hard_skills,soft_skills,education,job_category
0,4396160922,LinkedIn,https://www.linkedin.com/jobs/view/4396160922,Frontend Software Engineer,Frontend Software Engineer,Cabana,"Palo Alto, CA",2026-03-31,mid,2.0,full-time,onsite,135000.0,200000.0,167500.0,"JavaScript, TypeScript, frontend frameworks, web applications, frontend arch...","ownership, critical thinking, product sense, problem-solving, collaboration,...",unknown,software-eng
1,4395453046,LinkedIn,https://www.linkedin.com/jobs/view/4395453046,Software Engineer,Software Engineer,Epsilon,"Westminster, CO",2026-04-07,junior,1.0,full-time,unknown,73500.0,136500.0,105000.0,"Java, Python, SQL, relational databases, Unix, Unix Shell Scripts, Hadoop, H...","ownership, teamwork, collaboration, autonomy, growth mindset, continuous imp...",bachelor,software-eng
2,4364164472,LinkedIn,https://www.linkedin.com/jobs/view/4364164472,Software Engineer 5 - Ads Member Experience,Software Engineer 5 - Ads Member Experience,Netflix,"Los Gatos, CA",2026-03-29,senior,NaN,full-time,unknown,NaN,NaN,NaN,"ad formats, ad delivery systems, experimentation, ad creatives, native ads, ...","collaboration, creativity, innovation",unknown,software-eng
